1. One directional venue pair: where we buy and where we sell.
2. A fixed trade size, fees on both venues, and a minimum net price difference.
3. Episode start: the first order book event after which the condition is met.
4. Episode end: the first subsequent book event from either venue in the pair after which the condition fails.
5. A separate outcome: an order book becomes unavailable, a feed stays silent for too long, or the capture ends. Such an episode is neither confirmed to have ended nor known to still be active.

In [3]:
from pathlib import Path
import polars as pl


In [5]:
dataset = Path.cwd().parents[1] / 'experiments/data/parquet/6b556515-5156-4181-8a6f-7403fa9078fe'

for table in ("order_book_events", "availability_events"):
    files = [str(p) for p in dataset.glob(f"**/event_type={table}/**/*.parquet")]
    print(f"\n{table}: {len(files)} files")
    if files:
        data = pl.scan_parquet(files, hive_partitioning=True)
        print(data.collect_schema())
        print(data.head(3).collect())


order_book_events: 3273 files
Schema({'capture_id': String, 'capture_sequence': UInt64, 'venue': String, 'market_coin': String, 'local_receive_time': UInt64, 'processing_completion_time': UInt64, 'source_sequence': UInt64, 'bid_count': UInt32, 'ask_count': UInt32, 'version': Int64, 'date': Date, 'event_type': String, 'market': String})
shape: (3, 13)
┌─────────────┬────────────┬─────────┬────────────┬───┬─────────┬────────────┬────────────┬────────┐
│ capture_id  ┆ capture_se ┆ venue   ┆ market_coi ┆ … ┆ version ┆ date       ┆ event_type ┆ market │
│ ---         ┆ quence     ┆ ---     ┆ n          ┆   ┆ ---     ┆ ---        ┆ ---        ┆ ---    │
│ str         ┆ ---        ┆ str     ┆ ---        ┆   ┆ i64     ┆ date       ┆ str        ┆ str    │
│             ┆ u64        ┆         ┆ str        ┆   ┆         ┆            ┆            ┆        │
╞═════════════╪════════════╪═════════╪════════════╪═══╪═════════╪════════════╪════════════╪════════╡
│ 6b556515-51 ┆ 1875972    ┆ lighter ┆ B

In [7]:
level_files = [
    str(p) for p in dataset.glob("**/event_type=order_book_levels/**/*.parquet")
]
levels = pl.scan_parquet(level_files, hive_partitioning=True)

print("Order book level columns:")
print(levels.collect_schema())

Order book level columns:
Schema({'capture_id': String, 'capture_sequence': UInt64, 'venue': String, 'market_coin': String, 'side': String, 'position': UInt32, 'price': Decimal(precision=38, scale=18), 'quantity': Decimal(precision=38, scale=18), 'order_count': UInt32, 'version': Int64, 'date': Date, 'event_type': String, 'market': String})


In [ ]:
levels.head(3)